In [2]:
%pip install lightning tokenizers rdkit==2026.3.3


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Users/chandrakrajwal/.pyenv/versions/3.12.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from functools import partial

from sklearn.model_selection import train_test_split

from rdkit.Chem import MolFromSmiles, MolToSmiles, MolToInchiKey
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
import rdkit.rdBase as rkrb
import rdkit.RDLogger as rkl

import torch
from torch import nn
import torch.nn.functional as F
from torch.nn import TransformerEncoderLayer, TransformerEncoder
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from tokenizers import Tokenizer, models, trainers
from tokenizers.processors import TemplateProcessing

import lightning as L
from lightning.pytorch import loggers

tqdm.pandas()

In [4]:
COMP_DIR = 'enveda-CASMI26-molecule-id-mass-spectra'
TRAIN_PATH = os.path.join(COMP_DIR, 'train.parquet')
TEST_PATH = os.path.join(COMP_DIR, 'test.parquet')

In [6]:
SAMPLE_SUBMISSION_PATH = os.path.join(COMP_DIR, 'sample_submission.csv')
WORKING_DIR = '/working'
SUBMISSION_PATH = os.path.join(WORKING_DIR, 'submission.csv')

In [7]:
print(os.listdir(COMP_DIR))

['train.parquet', 'test.parquet', '.ipynb_checkpoints', 'sample_submission.csv']


In [8]:
logger = rkl.logger()
logger.setLevel(rkl.ERROR)
rkrb.DisableLog("rdApp.error")

#### LOADING DATA ...

In [10]:
NEEDED = ['ingest_lib', 'inchikey14', 'normalized_smiles', 'precursor_mz', 'ms2_mzs', 'ms2_normalized_intensities']
casmi_all_df = pd.read_parquet(TRAIN_PATH, columns=NEEDED, filters=[('ingest_lib', '!=', 'enveda-180')])

In [11]:
# split into train, val

all_structs = casmi_all_df.inchikey14.unique()
train_structs, val_structs = train_test_split(all_structs, test_size=500, random_state=0)
casmi_train_df = casmi_all_df[casmi_all_df.inchikey14.isin(train_structs)]
casmi_val_df = casmi_all_df[casmi_all_df.inchikey14.isin(val_structs)]

In [12]:
MAX_TRAIN_SPECTRA = 200_000
if MAX_TRAIN_SPECTRA is not None and len(casmi_train_df) > MAX_TRAIN_SPECTRA:
    casmi_train_df = casmi_train_df.sample(n=MAX_TRAIN_SPECTRA, random_state=0)

casmi_val_df = casmi_val_df.drop_duplicates(subset=['inchikey14'], keep='first')

print('split: spectra count, structure count')
print('train:', len(casmi_train_df), casmi_train_df.inchikey14.nunique())
print('val:', len(casmi_val_df), casmi_val_df.inchikey14.nunique())

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.
split: spectra count, structure count
train: 200000 57021
val: 500 500


In [13]:
MAX_LEN = 128
BPE_PAD_ID = 0
BPE_BOS_ID = 1
BPE_EOS_ID = 2
VOCAB_SIZE = 512

In [15]:
def process_spectra(df):
    def _process_spectrum(row):
        sort_mask = np.argsort(row.ms2_normalized_intensities)[::-1]
        sorted_mzs = row.ms2_mzs[sort_mask]
        sorted_intensities = row.ms2_normalized_intensities[sort_mask]

        truncated_mzs = sorted_mzs[:MAX_LEN - 1]
        truncated_intensities = sorted_intensities[:MAX_LEN - 1]
        normalized_intensities = truncated_intensities / truncated_intensities.max()

        return truncated_mzs, normalized_intensities

    mzs, ints = zip(*df.progress_apply(_process_spectrum, axis=1))
    df['processed_mzs'] = mzs
    df['processed_intensities'] = ints
    return df

print('processing spectra...')
casmi_train_df = process_spectra(casmi_train_df)
casmi_val_df = process_spectra(casmi_val_df)

def train_bpe_tokenizer(structures):
    os.environ["TOKENIZERS_PARALLELISM"] = "true"

    full_alphabet = list(set(list("".join(structures))))
    tokenizer = Tokenizer(models.BPE())
    special_tokens = [
        ("<pad>", BPE_PAD_ID),
        ("<s>", BPE_BOS_ID),
        ("</s>", BPE_EOS_ID),
    ]
    tokenizer.post_processor = TemplateProcessing(
        single="<s> $A </s>", special_tokens=special_tokens
    )
    special_tokens = ["<pad>", "<s>", "</s>"]

    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        initial_alphabet=full_alphabet,
        special_tokens=special_tokens,
        show_progress=True,
    )

    train_iterator = structures
    tokenizer.train_from_iterator(train_iterator, trainer)

    os.environ["TOKENIZERS_PARALLELISM"] = 'false'

    return tokenizer

def tokenizer_df_smiles(df, tokenizer):
    df['bpe_tokenized_smiles'] = df.normalized_smiles.progress_apply(
        lambda x: np.array(tokenizer.encode(x).ids, dtype=int)
    )
    return df

print('training bpe...')
bpe_train_structures = list(casmi_train_df.normalized_smiles.unique())
tokenizer = train_bpe_tokenizer(bpe_train_structures)

print('processing smiles...')
casmi_train_df = tokenizer_df_smiles(casmi_train_df, tokenizer=tokenizer)
casmi_val_df = tokenizer_df_smiles(casmi_val_df, tokenizer=tokenizer)


def decode_tokenized_smiles(bpe_smiles_tokens, tokenizer=tokenizer):
    return tokenizer.decode(bpe_smiles_tokens).replace(' ', '')



    

processing spectra...


100%|██████████████████████████████████████| 500/500 [00:00<00:00, 28237.63it/s]


training bpe...



processing smiles...


100%|██████████████████████████████████████| 500/500 [00:00<00:00, 58926.97it/s]


### Data Module Class

In [18]:
PRECURSOR_INTENSITY = 2.0
MZ_PAD_ID = 0
INTENSITY_PAD_ID = 0
BATCH_SIZE = 128
NUM_WORKERS = 3

In [19]:
class PandasDataset(Dataset):
    def __init__(self, df, column_list, shuffle=True):

        self.df = df[column_list].sample(frac=1, random_state=0) if shuffle else df[column_list]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self.df.iloc[idx].to_dict()

In [20]:
class CASMIDataModule(L.LightningDataModule):

    def __init__(self, batch_size, predict_batch_size=None):
        super().__init__()
        self.batch_size = batch_size
        # Generation expands the batch by n_samples, so prediction needs a far smaller batch than
        # training. See the predict cell for the arithmetic.
        self.predict_batch_size = predict_batch_size or batch_size

    def setup(self, stage):
        columns = ['precursor_mz', 'processed_mzs', 'processed_intensities', 'bpe_tokenized_smiles', 'normalized_smiles']
        if stage == "fit":
            self.train_df = PandasDataset(casmi_train_df, column_list=columns)
            self.val_df = PandasDataset(casmi_val_df, column_list=columns)
        elif stage == "validate":
            self.val_df = PandasDataset(casmi_val_df, column_list=columns)
        elif stage == "predict":
            # test.parquet carries no labels, so only the spectrum columns exist. molecule_id rides
            # along because scoring is per molecule, not per spectrum, and a molecule's spectra have
            # to be grouped back together afterwards.
            self.predict_df = PandasDataset(
                casmi_test_df,
                column_list=['molecule_id', 'precursor_mz', 'processed_mzs', 'processed_intensities'],
                shuffle=False,
            )

    def train_dataloader(self):
        return DataLoader(
            self.train_df,
            batch_size=self.batch_size,
            collate_fn=self.get_collator('fit'),
            num_workers=NUM_WORKERS,
            persistent_workers=NUM_WORKERS > 0,
            pin_memory=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_df,
            batch_size=self.batch_size,
            collate_fn=self.get_collator('validate'),
            num_workers=NUM_WORKERS,
            persistent_workers=NUM_WORKERS > 0,
            pin_memory=True,
        )

    def predict_dataloader(self):
        return DataLoader(
            self.predict_df,
            batch_size=self.predict_batch_size,
            collate_fn=self.get_collator('predict'),
            num_workers=NUM_WORKERS,
            persistent_workers=NUM_WORKERS > 0,
            pin_memory=True,
        )

    def get_collator(self, stage):
        return partial(self._collator, stage=stage)

    def _collator(self, data, stage='fit'):
        mzs = [[row['precursor_mz']] + row['processed_mzs'].tolist() for row in data]  # prepend precursor to m/zs
        ints = [[PRECURSOR_INTENSITY] + row['processed_intensities'].tolist() for row in data]  # prepend 2. to intensities
        if stage == 'fit' or stage == 'validate':
            labels = [row['bpe_tokenized_smiles'] for row in data]

        max_peak_len = max([len(x) for x in mzs])
        mzs = [list(x) + [MZ_PAD_ID] * (max_peak_len - len(x)) for x in mzs]
        ints = [list(x) + [INTENSITY_PAD_ID] * (max_peak_len - len(x)) for x in ints]
        mz_array = torch.tensor(mzs)
        intensity_array = torch.tensor(ints)
        attention_mask = torch.where(mz_array == MZ_PAD_ID, 0, 1)

        # collate labels
        batch = {}
        if stage == 'fit' or stage == 'validate':
            max_smiles_len = max([len(x) for x in labels])
            labels = [list(x) + [BPE_PAD_ID] * (max_smiles_len - len(x)) for x in labels]
            label_array = torch.tensor(labels)

            batch['mzs'] = mz_array
            batch['intensities'] = intensity_array
            batch['attention_mask'] = attention_mask
            batch['structure_tokens'] = label_array # tokenized labels for training
        if stage == 'validate':
            batch['smiles'] = [row['normalized_smiles'] for row in data] # include original smiles for validation metrics
        else:
            batch = batch | {
                'mzs': mz_array,
                'intensities': intensity_array,
                'attention_mask': attention_mask,
            }
        if stage == 'predict':
            batch['molecule_id'] = [row['molecule_id'] for row in data]
        return batch

datamodule = CASMIDataModule(batch_size=BATCH_SIZE)  
datamodule.setup('fit')



In [ ]:
val_dl = datamodule.val_dataloader()
for batch in val_dl:
    break
    
batch

/Users/chandrakrajwal/.pyenv/versions/3.12.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/chandrakrajwal/.pyenv/versions/3.12.12/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/chandrakrajwal/.pyenv/versions/3.12.12/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'PandasDataset' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
